# Raw to Bronze load

Loads landed files into the four Bronze tables. **Normal run: Run all with the default widgets.** Backfill: set `source_dir=processed` and a date or batch range. Replay from the start: `replay=true`. Details in the guide, sections 2A and 8.

### CELL 1 - Parameters and configuration

In [ ]:
# CELL 1 - Parameters and configuration
import csv, hashlib, json, os, re, tarfile, time, traceback, uuid
from datetime import date, datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (BooleanType, ByteType, DateType, DoubleType, FloatType, IntegerType,
                               LongType, ShortType, StringType, StructField, StructType, TimestampType)

# Parameters. A Databricks Job passes these as job parameters; the defaults are the normal scheduled (Mon/Wed/Fri) run.
dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("bronze_schema", "bronze", "bronze_schema (use bronze_test for the proof notebook)")
dbutils.widgets.dropdown("source", "all", ["all", "arena", "cooperbench", "benchlm", "epoch"], "source")
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"], "load_type")
dbutils.widgets.text("source_dir", "new", "source_dir: new | processed | /Volumes/...")
dbutils.widgets.text("batch_id", "", "batch_id (one batch; empty = every batch)")
dbutils.widgets.text("from_batch", "", "from_batch (first batch id or prefix, e.g. 20261005T000000Z)")
dbutils.widgets.text("to_batch", "", "to_batch (last batch id or prefix, inclusive)")
dbutils.widgets.text("start_date", "", "start_date YYYY-MM-DD (empty = no limit)")
dbutils.widgets.text("end_date", "", "end_date YYYY-MM-DD (empty = no limit)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "force_reload")
dbutils.widgets.dropdown("replay", "false", ["false", "true"],
                         "replay: rebuild from the first batch, one batch at a time, oldest first")

P = {k: dbutils.widgets.get(k).strip() for k in
     ["catalog", "bronze_schema", "source", "load_type", "source_dir", "batch_id", "from_batch", "to_batch",
      "start_date", "end_date", "force_reload", "replay"]}

# A bad parameter must not crash the notebook silently: it is collected here and written to the
# execution log (and error log) as a FAILURE row by run_with_log() in Cell 11.
PARAM_ERRORS = []


def parse_day(name):
    value = P[name]
    if not value:
        return ""
    try:
        return date.fromisoformat(value).isoformat()
    except ValueError:
        PARAM_ERRORS.append(f"{name}='{value}' is not a valid YYYY-MM-DD date")
        return ""


START, END = parse_day("start_date"), parse_day("end_date")
if START and END and START > END:
    PARAM_ERRORS.append(f"start_date {START} is after end_date {END}")

# Batch range: from_batch / to_batch take a full batch id or just its timestamp prefix (inclusive on both ends).
FROM_B, TO_B = P["from_batch"], P["to_batch"]
if FROM_B and TO_B:
    n = min(len(FROM_B), len(TO_B))
    if FROM_B[:n] > TO_B[:n]:
        PARAM_ERRORS.append(f"from_batch {FROM_B} is later than to_batch {TO_B}")

# REPLAY = rebuild Bronze from the very first batch, but NOT as one full load: every batch that arrived earlier is
# loaded again on its own, oldest first, with its own log rows. It reads from `processed` unless a folder is given.
# The MERGE on record_hash makes this safe, so a replay is always run with force_reload.
REPLAY = P["replay"] == "true"
if REPLAY and P["source_dir"] == "new":
    P["source_dir"] = "processed"
FORCE = P["force_reload"] == "true" or REPLAY

CATALOG, BSCHEMA = P["catalog"], P["bronze_schema"]
for _name in (CATALOG, BSCHEMA):                      # these two end up inside SQL text, so keep them plain
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", _name):
        raise ValueError(f"Invalid catalog/schema name: {_name!r}")
ROOT = f"/Volumes/{CATALOG}/landing/staging/frontier"
SRC_DIR = P["source_dir"].rstrip("/") if P["source_dir"].startswith("/") else f"{ROOT}/{P['source_dir']}"
IS_LIVE = SRC_DIR == f"{ROOT}/new"          # only the live inbox is emptied after loading

# Strict volume rules (checked again here, not only in the collector)
MIN_FULL_BYTES = 400_000_000
MIN_INCREMENTAL_BYTES = 1_000_000
MEMBER_EXTENSIONS = {".json", ".jsonl"}     # must match the collector
MAX_MEMBER_BYTES = 64_000_000               # bigger archive members are quarantined
BATCH_BYTES = 16_000_000                    # archive members are written in chunks of about this size (stays far below the Spark Connect message limit)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
spark.conf.set("spark.sql.session.timeZone", "UTC")

T_RUNS = f"{CATALOG}.ops.pipeline_execution_logs"
T_ERRORS = f"{CATALOG}.ops.error_logs"
T_QUAR = f"{CATALOG}.ops.quarantine_log"
T_DRIFT = f"{CATALOG}.ops.schema_drift_log"
T_ARENA = f"{CATALOG}.{BSCHEMA}.arena_leaderboard_raw"
T_COOP = f"{CATALOG}.{BSCHEMA}.cooperbench_member_raw"
T_BLM = f"{CATALOG}.{BSCHEMA}.benchlm_raw"
T_EPOCH = f"{CATALOG}.{BSCHEMA}.epoch_models_raw"

print("Run id:", RUN_ID, "| parameters:", P, "| reading from:", SRC_DIR)

### CELL 2 - Explicit schemas (the data contracts). Nothing is inferred anywhere in this notebook.

In [ ]:
# CELL 2 - Explicit schemas (the data contracts). Nothing is inferred anywhere in this notebook.

# Arena Parquet: expected columns. If the smoke test printed something different, edit this list.
ARENA_CONTRACT = StructType([
    StructField("model_name", StringType(), True),
    StructField("organization", StringType(), True),
    StructField("license", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("variance", DoubleType(), True),
    StructField("rating_q975", DoubleType(), True),
    StructField("rating_q025", DoubleType(), True),
    StructField("vote_count", LongType(), True),
    StructField("rank", LongType(), True),
    StructField("category", StringType(), True),
    StructField("leaderboard_publish_date", StringType(), True),
])
ARENA_META = ["subset", "load_type", "raw_payload", "record_hash", "source_file", "batch_id",
              "collection_timestamp", "load_timestamp"]

# Epoch CSV: Bronze keeps every value as text, so the contract fixes the NAMES of the core columns the
# source must keep (all StringType). Extend this list from the smoke-test output of your real file.
# A core column that disappears is written to ops.schema_drift_log (MISSING_COLUMN); a column the contract
# does not know is read as text too and logged (NEW_COLUMN). Neither stops the load.
EPOCH_CONTRACT = StructType([StructField(n, StringType(), True) for n in [
    "Model", "Domain", "Authors", "Link", "Citations", "Reference", "Publication date", "Organization",
    "Parameters", "Training compute (FLOP)"]])

# Rows built in Python for the archive members
COOP_ROW_SCHEMA = StructType([
    StructField("study_setting", StringType(), True), StructField("archive_file", StringType(), True),
    StructField("member_path", StringType(), True), StructField("member_extension", StringType(), True),
    StructField("member_size_bytes", LongType(), True), StructField("raw_content", StringType(), True),
    StructField("record_hash", StringType(), True), StructField("ingested_at", StringType(), True)])

BLM_ROW_SCHEMA = StructType([
    StructField("dataset", StringType(), True), StructField("record_path", StringType(), True),
    StructField("record_key", StringType(), True), StructField("record_index", LongType(), True),
    StructField("raw_payload", StringType(), True), StructField("payload_hash", StringType(), True),
    StructField("record_hash", StringType(), True), StructField("ingested_at", StringType(), True)])

SPARK_TYPES = {"string": StringType(), "bigint": LongType(), "int": IntegerType(), "smallint": ShortType(),
               "tinyint": ByteType(), "double": DoubleType(), "float": FloatType(),
               "boolean": BooleanType(), "timestamp": TimestampType(), "date": DateType()}
# (type in the file, type in the table) pairs that are safe to read without loss
WIDENING_OK = {("tinyint", "smallint"), ("tinyint", "int"), ("tinyint", "bigint"), ("smallint", "int"),
               ("smallint", "bigint"), ("int", "bigint"), ("float", "double")}

### CELL 3 - Create the tables (every table has load_timestamp)

In [ ]:
# CELL 3 - Create the tables (every table has load_timestamp)

OPS_SPECS = {
    T_RUNS: [("run_id", "string"), ("batch_id", "string"), ("layer", "string"), ("source", "string"),
             ("dataset", "string"), ("load_type", "string"), ("file_or_param", "string"),
             ("parameters_json", "string"), ("start_time", "timestamp"), ("end_time", "timestamp"),
             ("duration_seconds", "double"), ("status", "string"), ("rows_read", "bigint"),
             ("rows_inserted", "bigint"), ("rows_updated", "bigint"), ("rows_skipped_duplicate", "bigint"),
             ("rows_quarantined", "bigint"), ("bytes_processed", "bigint"), ("message", "string"),
             ("load_timestamp", "timestamp")],
    T_ERRORS: [("error_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("component", "string"),
               ("layer", "string"), ("source", "string"), ("file_or_param", "string"), ("severity", "string"),
               ("error_type", "string"), ("error_message", "string"), ("http_status", "bigint"),
               ("attempt", "bigint"), ("stack_trace", "string"), ("occurred_at", "timestamp"),
               ("resolved", "boolean"), ("load_timestamp", "timestamp")],
    T_QUAR: [("quarantine_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("source", "string"),
             ("dataset", "string"), ("source_file", "string"), ("item", "string"), ("reason", "string"),
             ("detail", "string"), ("size_bytes", "bigint"), ("record_sample", "string"),
             ("quarantined_at", "timestamp"), ("load_timestamp", "timestamp")],
    T_DRIFT: [("drift_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("dataset", "string"),
              ("source_file", "string"), ("column_name", "string"), ("event_type", "string"),
              ("expected_type", "string"), ("actual_type", "string"), ("action_taken", "string"),
              ("detected_at", "timestamp"), ("load_timestamp", "timestamp")],
}
OPS_TYPES = {"string": StringType(), "bigint": LongType(), "double": DoubleType(), "boolean": BooleanType()}

for schema in (BSCHEMA, "ops"):                      # the notebook runs from scratch: no manual setup needed
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for table, spec in OPS_SPECS.items():
    ddl = ", ".join(f"`{n}` {t.upper()}" for n, t in spec)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({ddl}) USING DELTA")
    have = {c.lower() for c in spark.table(table).columns}          # an older table gets the new columns
    for n, t in spec:
        if n.lower() not in have:
            spark.sql(f"ALTER TABLE {table} ADD COLUMNS (`{n}` {t.upper()})")

arena_cols = ", ".join(f"`{f.name}` {f.dataType.simpleString().upper()}" for f in ARENA_CONTRACT.fields)
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_ARENA} (
    subset STRING, load_type STRING, {arena_cols},
    raw_payload STRING, record_hash STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_COOP} (
    study_setting STRING, archive_file STRING, member_path STRING, member_extension STRING,
    member_size_bytes BIGINT, raw_content STRING, record_hash STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_BLM} (
    dataset STRING, record_path STRING, record_key STRING, record_index BIGINT, raw_payload STRING,
    payload_hash STRING, record_hash STRING, snapshot_id STRING, source_file STRING, batch_id STRING,
    collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_EPOCH} (
    raw_payload STRING, payload_hash STRING, record_hash STRING, snapshot_id STRING, source_file STRING,
    batch_id STRING, collection_timestamp TIMESTAMP, load_timestamp TIMESTAMP) USING DELTA""")

print("Tables ready.")

### CELL 4 - Logging helpers (execution log, error log, quarantine log, drift log)

In [ ]:
# CELL 4 - Logging helpers (execution log, error log, quarantine log, drift log)

def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def append_record(table, record):
    """Appends one row to an ops table. Timestamps are passed as ISO text and converted here."""
    spec = OPS_SPECS[table]
    schema = StructType([StructField(n, StringType() if t == "timestamp" else OPS_TYPES[t], True) for n, t in spec])
    df = spark.createDataFrame([tuple(record.get(n) for n, _ in spec)], schema)
    for name, kind in spec:
        if kind == "timestamp":
            df = df.withColumn(name, F.to_timestamp(F.col(name)))
    df.write.mode("append").saveAsTable(table)


def log_error(stage, source, file_or_param, message, severity="ERROR", error_type="", batch_id="",
              stack="", http_status=None, attempt=None, component="bronze_notebook", occurred_at=None):
    print(f"{severity} | {stage} | {source} | {file_or_param} | {message}")
    try:
        append_record(T_ERRORS, {
            "error_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id, "component": component,
            "layer": stage, "source": source, "file_or_param": file_or_param, "severity": severity,
            "error_type": error_type, "error_message": str(message)[:4000], "http_status": http_status,
            "attempt": attempt, "stack_trace": stack[-4000:], "occurred_at": occurred_at or now_iso(),
            "resolved": False, "load_timestamp": now_iso()})
    except Exception as exc:            # logging must never hide the original problem
        print("Could not write to the error table:", exc)


def log_quarantine(batch_id, source, dataset, source_file, item, reason, detail="", size_bytes=None, sample=""):
    print(f"QUARANTINE | {source} | {item} | {reason}")
    append_record(T_QUAR, {"quarantine_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id,
                           "source": source, "dataset": dataset, "source_file": source_file, "item": item,
                           "reason": reason, "detail": str(detail)[:2000], "size_bytes": size_bytes,
                           "record_sample": str(sample)[:4000],
                           "quarantined_at": now_iso(), "load_timestamp": now_iso()})


def log_drift(batch_id, dataset, source_file, column, event, expected, actual, action):
    append_record(T_DRIFT, {"drift_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id,
                            "dataset": dataset, "source_file": source_file, "column_name": column,
                            "event_type": event, "expected_type": expected, "actual_type": actual,
                            "action_taken": action, "detected_at": now_iso(), "load_timestamp": now_iso()})


class QuarantineFile(Exception):
    """Raised when a whole file does not match its contract. The batch carries on without it.
    `sample` is stored in ops.quarantine_log.record_sample so the bad data can be inspected later."""
    def __init__(self, message, sample=""):
        super().__init__(message)
        self.sample = sample


# Totals of the file-level steps of this run (written into the run-level log row)
RUN_TOTALS = {"rows_read": 0, "rows_inserted": 0, "rows_updated": 0, "rows_skipped": 0,
              "rows_quarantined": 0, "bytes": 0, "outcomes": {}}


class Step:
    """One row of ops.pipeline_execution_logs per file or table processed."""
    def __init__(self, layer, source, dataset, load_type, file_or_param, batch_id, extra=None):
        self.layer, self.source, self.dataset, self.load_type = layer, source, dataset, load_type
        self.file_or_param, self.batch_id = file_or_param, batch_id
        self.params = dict(P, **(extra or {}))
        self.start = now_iso()
        self.t0 = time.time()
        self.rows_read = self.rows_inserted = self.rows_updated = 0
        self.rows_skipped = self.rows_quarantined = self.bytes = 0

    def finish(self, status, message=""):
        append_record(T_RUNS, {
            "run_id": RUN_ID, "batch_id": self.batch_id, "layer": self.layer, "source": self.source,
            "dataset": self.dataset, "load_type": self.load_type, "file_or_param": self.file_or_param,
            "parameters_json": json.dumps(self.params), "start_time": self.start, "end_time": now_iso(),
            "duration_seconds": round(time.time() - self.t0, 2), "status": status,
            "rows_read": self.rows_read, "rows_inserted": self.rows_inserted, "rows_updated": self.rows_updated,
            "rows_skipped_duplicate": self.rows_skipped, "rows_quarantined": self.rows_quarantined,
            "bytes_processed": self.bytes, "message": str(message)[:2000], "load_timestamp": now_iso()})
        if self.layer == "Raw-to-Bronze":                 # file-level steps only (not the batch/run rows)
            for key, value in (("rows_read", self.rows_read), ("rows_inserted", self.rows_inserted),
                               ("rows_updated", self.rows_updated), ("rows_skipped", self.rows_skipped),
                               ("rows_quarantined", self.rows_quarantined), ("bytes", self.bytes)):
                RUN_TOTALS[key] += value
        print(f"{status} | {self.layer} | {self.file_or_param} | read {self.rows_read} | inserted "
              f"{self.rows_inserted} | updated {self.rows_updated} | skipped {self.rows_skipped}")

### CELL 5 - Helper functions

In [ ]:
# CELL 5 - Helper functions

def local(path):
    return path.replace("dbfs:", "", 1)


def exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def sha256_file(path):
    digest = hashlib.sha256()
    with open(local(path), "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sql_id(name):
    return "`" + name.replace("`", "``") + "`"


def move(src, dst):
    if exists(dst):
        dbutils.fs.rm(dst)
    dbutils.fs.mv(src, dst)


def merge_insert_only(df, target, key="record_hash"):
    """MERGE INTO: a record whose key already exists is left alone, a new key is inserted.
    Running the same data twice therefore inserts nothing the second time."""
    view = "src_" + uuid.uuid4().hex[:10]
    df.createOrReplaceTempView(view)
    cols = df.columns
    insert_cols = ", ".join(sql_id(c) for c in cols)
    insert_vals = ", ".join("s." + sql_id(c) for c in cols)
    result = spark.sql(f"""
        MERGE INTO {target} AS t USING {view} AS s ON t.{key} = s.{key}
        WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})""").collect()[0].asDict()
    spark.catalog.dropTempView(view)
    return int(result.get("num_inserted_rows") or 0), int(result.get("num_updated_rows") or 0)


def batch_columns(df, meta):
    """Adds the columns every Bronze row carries."""
    out = (df.withColumn("source_file", F.lit(meta["source_file"]))
              .withColumn("batch_id", F.lit(meta["batch_id"]))
              .withColumn("collection_timestamp", F.to_timestamp(F.lit(meta["created_at"]))))
    if "load_timestamp" not in df.columns:               # the moment this batch is written to Bronze
        out = out.withColumn("load_timestamp", F.current_timestamp())
    return out


def arrow_type_name(arrow_type):
    import pyarrow as pa
    t = pa.types
    if t.is_dictionary(arrow_type):
        return arrow_type_name(arrow_type.value_type)
    for check, name in [(t.is_string, "string"), (t.is_large_string, "string"), (t.is_int64, "bigint"),
                        (t.is_int32, "int"), (t.is_int16, "smallint"), (t.is_int8, "tinyint"),
                        (t.is_uint8, "smallint"), (t.is_uint16, "int"), (t.is_uint32, "bigint"),
                        (t.is_float64, "double"), (t.is_float32, "float"), (t.is_boolean, "boolean"),
                        (t.is_date32, "date")]:
        if check(arrow_type):
            return name
    if t.is_timestamp(arrow_type):
        return "timestamp"
    return None                              # lists, structs, decimals, ... not supported in Bronze columns


def parquet_columns(path):
    import pyarrow.parquet as pq
    return {f.name: arrow_type_name(f.type) for f in pq.read_schema(local(path))}


def arena_prepare(path, meta):
    """Compares the file with the table. New column -> table evolves. Type conflict -> file quarantined.
    Returns (read_schema, output_columns): the explicit schema to read the file with (the types the file
    really has) and the typed columns to write (the types the table has)."""
    batch, dataset, rel = meta["batch_id"], f"arena/{meta['dataset']}", meta["source_file"]
    table_fields = [f for f in spark.table(T_ARENA).schema.fields if f.name not in ARENA_META]
    expected = {f.name.lower(): (f.name, f.dataType.simpleString()) for f in table_fields}
    actual = {name.lower(): (name, kind) for name, kind in parquet_columns(path).items()}

    problems, new_cols = [], []
    for low, (name, kind) in actual.items():
        if low not in expected:
            if kind is None:
                problems.append(f"{name}: unsupported type")
                log_drift(batch, dataset, rel, name, "UNSUPPORTED_TYPE", "", "complex", "FILE_QUARANTINED")
            else:
                new_cols.append((name, kind))
        else:
            want = expected[low][1]
            if kind != want and (kind, want) not in WIDENING_OK:
                problems.append(f"{name}: table has {want}, file has {kind}")
                log_drift(batch, dataset, rel, name, "TYPE_CHANGE", want, str(kind), "FILE_QUARANTINED")
    missing = [expected[low][0] for low in expected if low not in actual]
    for name in missing:
        log_drift(batch, dataset, rel, name, "MISSING_COLUMN", expected[name.lower()][1], "", "FILLED_WITH_NULL")
    if len(missing) > len(expected) / 2:
        problems.append(f"{len(missing)} of {len(expected)} expected columns are missing")
    if problems:
        raise QuarantineFile("; ".join(problems),
                             sample="file columns: " + ", ".join(f"{n}:{k}" for n, k in actual.values()))

    for name, kind in new_cols:                                    # schema evolution
        spark.sql(f"ALTER TABLE {T_ARENA} ADD COLUMNS ({sql_id(name)} {kind.upper()})")
        log_drift(batch, dataset, rel, name, "NEW_COLUMN", "", kind, "TABLE_EVOLVED")

    # Contract first: a column the table (ARENA_CONTRACT) already knows is read with the contract type.
    # Only a safe widening (float in the file, double in the table) is read as the file has it and cast after.
    contract_types = {f.name.lower(): f.dataType for f in table_fields}
    read_schema = StructType([
        StructField(name, contract_types[name.lower()] if name.lower() in expected and kind == expected[name.lower()][1]
                    else SPARK_TYPES[kind], True)
        for name, kind in actual.values()])
    outputs = []
    for f in table_fields:
        if f.name.lower() in actual:       # compatible type: read as the file has it, cast to the table type
            outputs.append(F.col(sql_id(actual[f.name.lower()][0])).cast(f.dataType).alias(f.name))
        else:
            outputs.append(F.lit(None).cast(f.dataType).alias(f.name))
    for name, kind in new_cols:
        outputs.append(F.col(sql_id(name)).alias(name))
    return read_schema, outputs

### CELL 6 - Pick up the collector's logs (GitHub -> Databricks) so every error is in one table

In [ ]:
# CELL 6 - Pick up the collector's logs (GitHub -> Databricks) so every error is in one table

def ingest_collector_logs():
    pending, done = f"{ROOT}/logs/pending", f"{ROOT}/logs/ingested"
    if not exists(pending):
        return
    for f in dbutils.fs.ls(pending):
        try:
            with open(local(f.path), encoding="utf-8") as handle:
                if f.name.startswith("collector_errors_") and f.name.endswith(".jsonl"):
                    for line in handle:
                        if not line.strip():
                            continue
                        e = json.loads(line)
                        append_record(T_ERRORS, {
                            "error_id": e["error_id"], "run_id": RUN_ID, "batch_id": e.get("batch_id"),
                            "component": "collector", "layer": e.get("stage"), "source": e.get("source"),
                            "file_or_param": e.get("file_or_param"), "severity": e.get("severity"),
                            "error_type": e.get("error_type"), "error_message": e.get("message"),
                            "http_status": e.get("http_status"), "attempt": e.get("attempt"),
                            "stack_trace": e.get("traceback"), "occurred_at": e.get("occurred_at"),
                            "resolved": False, "load_timestamp": now_iso()})
                elif f.name.startswith("collector_run_") and f.name.endswith(".json"):
                    s = json.load(handle)
                    append_record(T_RUNS, {
                        "run_id": s["run_id"], "batch_id": s["batch_id"], "layer": "Source-to-Landing",
                        "source": "all", "dataset": "all", "load_type": s["mode"],
                        "file_or_param": f"github-actions:{s['mode']}", "parameters_json": json.dumps(s),
                        "start_time": s["started_at"], "end_time": s["finished_at"],
                        "duration_seconds": None, "status": s["status"], "rows_read": 0, "rows_inserted": 0,
                        "rows_updated": 0, "rows_skipped_duplicate": 0, "rows_quarantined": 0,
                        "bytes_processed": s.get("downloaded_bytes", 0), "message": s.get("message", ""),
                        "load_timestamp": now_iso()})
                else:
                    continue
            move(f.path, f"{done}/{f.name}")
            print("Collector log ingested:", f.name)
        except Exception as exc:
            log_error("ingest_collector_logs", "collector", f.name, exc, error_type=type(exc).__name__,
                      stack=traceback.format_exc())

### CELL 7 - Loader: Arena Parquet -> bronze.arena_leaderboard_raw

In [ ]:
# CELL 7 - Loader: Arena Parquet -> bronze.arena_leaderboard_raw

def load_arena(entry, meta, path, step):
    read_schema, outputs = arena_prepare(path, meta)             # explicit schema, never inferred
    raw = spark.read.schema(read_schema).parquet(path)
    step.rows_read = raw.count()
    fields = sorted(raw.columns, key=str.lower)
    df = (raw.withColumn("raw_payload", F.to_json(F.struct(*[F.col(sql_id(c)) for c in fields])))
             .withColumn("subset", F.lit(meta["dataset"]))
             .withColumn("load_type", F.lit(meta["load_type"])))
    df = df.withColumn("record_hash", F.sha2(F.concat_ws("|", F.col("subset"), F.col("raw_payload")), 256))
    df = df.dropDuplicates(["record_hash"])
    df = df.select("subset", "load_type", *outputs, "raw_payload", "record_hash")
    df = batch_columns(df, meta)
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_ARENA)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 8 - Loader: CooperBench archives -> bronze.cooperbench_member_raw

In [ ]:
# CELL 8 - Loader: CooperBench archives -> bronze.cooperbench_member_raw

def load_cooperbench(entry, meta, path, step):
    setting, archive = meta["dataset"], os.path.basename(meta["source_file"])
    rows, chunk_bytes, ignored = [], 0, 0

    def flush():
        nonlocal rows, chunk_bytes
        if not rows:
            return
        df = spark.createDataFrame(rows, COOP_ROW_SCHEMA).drop("ingested_at")
        df = batch_columns(df.dropDuplicates(["record_hash"]), meta)       # load_timestamp = write time
        df = df.select("study_setting", "archive_file", "member_path", "member_extension", "member_size_bytes",
                       "raw_content", "record_hash", "source_file", "batch_id", "collection_timestamp",
                       "load_timestamp")
        inserted, updated = merge_insert_only(df, T_COOP)
        step.rows_inserted += inserted
        step.rows_updated += updated
        rows, chunk_bytes = [], 0

    with tarfile.open(local(path), "r|gz") as tar:               # streamed: the archive is never unpacked to disk
        for member in tar:
            if not member.isfile():
                continue
            ext = os.path.splitext(member.name)[1].lower()
            if ext not in MEMBER_EXTENSIONS:
                ignored += 1
                continue
            if member.size > MAX_MEMBER_BYTES:
                log_quarantine(meta["batch_id"], "cooperbench", setting, meta["source_file"], member.name,
                               "MEMBER_TOO_LARGE", f"{member.size} bytes", member.size,
                               sample="content not read (larger than MAX_MEMBER_BYTES)")
                step.rows_quarantined += 1
                continue
            if rows and chunk_bytes + member.size > BATCH_BYTES:         # keep every chunk small
                flush()
            data = tar.extractfile(member).read()
            try:
                text = data.decode("utf-8")
            except UnicodeDecodeError as exc:
                log_quarantine(meta["batch_id"], "cooperbench", setting, meta["source_file"], member.name,
                               "NOT_UTF8", exc, member.size, sample="first 200 bytes (hex): " + data[:200].hex())
                step.rows_quarantined += 1
                continue
            content_hash = hashlib.sha256(data).hexdigest()
            record_hash = hashlib.sha256(f"{setting}|{member.name}|{content_hash}".encode()).hexdigest()
            rows.append((setting, archive, member.name, ext, member.size, text, record_hash, now_iso()))
            step.rows_read += 1
            step.bytes += member.size
            chunk_bytes += member.size
            if chunk_bytes >= BATCH_BYTES:
                flush()
    flush()
    step.rows_skipped = step.rows_read - step.rows_inserted
    step.params["non_json_members_ignored"] = ignored

### CELL 9 - Loader: BenchLM JSON -> bronze.benchlm_raw

In [ ]:
# CELL 9 - Loader: BenchLM JSON -> bronze.benchlm_raw

def split_records(doc):
    """Finds the list of records inside a JSON export. Returns (path, [(key, record), ...])."""
    if isinstance(doc, list):
        return "$", [(None, r) for r in doc]
    if isinstance(doc, dict):
        lists = [(k, v) for k, v in doc.items() if isinstance(v, list) and v and all(isinstance(x, dict) for x in v)]
        if lists:
            key, items = max(lists, key=lambda kv: len(kv[1]))
            return "$." + key, [(None, r) for r in items]
        if len(doc) > 1 and all(isinstance(v, dict) for v in doc.values()):
            return "$", [(k, v) for k, v in doc.items()]
    return "$", [(None, doc)]


def track_columns(batch, dataset, source_file, columns):
    """Payload-only sources never change the table; new fields are simply recorded in the drift log."""
    known = {r[0] for r in spark.sql(
        f"SELECT column_name FROM {T_DRIFT} WHERE dataset = '{dataset}' "
        f"AND event_type IN ('BASELINE','NEW_COLUMN')").collect()}
    for name in sorted(set(columns) - known):
        log_drift(batch, dataset, source_file, name, "BASELINE" if not known else "NEW_COLUMN", "", "json/string",
                  "ABSORBED_IN_RAW_PAYLOAD")


def load_benchlm(entry, meta, path, step):
    with open(local(path), encoding="utf-8") as handle:
        doc = json.load(handle)
    record_path, records = split_records(doc)
    snapshot = entry["sha256"]
    rows, columns = [], set()
    for index, (key, record) in enumerate(records):
        payload = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        if isinstance(record, dict):
            columns.update(record.keys())
        rows.append((meta["dataset"], record_path, key, index, payload,
                     hashlib.sha256(payload.encode()).hexdigest(),
                     hashlib.sha256(f"{snapshot}|{meta['dataset']}|{index}".encode()).hexdigest(), now_iso()))
    step.rows_read = len(rows)
    track_columns(meta["batch_id"], f"benchlm/{meta['dataset']}", meta["source_file"], columns)
    df = spark.createDataFrame(rows, BLM_ROW_SCHEMA)
    df = batch_columns(df.drop("ingested_at"), meta).withColumn("snapshot_id", F.lit(snapshot))
    df = df.select("dataset", "record_path", "record_key", "record_index", "raw_payload", "payload_hash",
                   "record_hash", "snapshot_id", "source_file", "batch_id", "collection_timestamp", "load_timestamp")
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_BLM)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 10 - Loader: Epoch CSV -> bronze.epoch_models_raw

In [ ]:
# CELL 10 - Loader: Epoch CSV -> bronze.epoch_models_raw

def load_epoch(entry, meta, path, step):
    with open(local(path), encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        python_rows = sum(1 for _ in reader)
    if len(set(header)) != len(header):
        raise QuarantineFile("The CSV header has duplicate column names")
    contract_types = {f.name.lower(): f.dataType for f in EPOCH_CONTRACT.fields}
    for f in EPOCH_CONTRACT.fields:                                   # core column gone -> logged, load continues
        if f.name.lower() not in {h.lower() for h in header}:
            log_drift(meta["batch_id"], "epoch/all_ai_models", meta["source_file"], f.name, "MISSING_COLUMN",
                      "string", "", "ABSORBED_IN_RAW_PAYLOAD")
    # Read schema: contract type for every core column, text for any new one, in the file's own column order
    schema = StructType([StructField(name, contract_types.get(name.lower(), StringType()), True) for name in header])
    raw = (spark.read.schema(schema).option("header", "true").option("multiLine", "true")
           .option("quote", '"').option("escape", '"').csv(path))
    step.rows_read = raw.count()
    if step.rows_read != python_rows:
        log_error("Raw-to-Bronze", "epoch", meta["source_file"],
                  f"Spark read {step.rows_read} rows but a second parser counted {python_rows}",
                  severity="WARNING", error_type="RowCountMismatch", batch_id=meta["batch_id"])
    track_columns(meta["batch_id"], "epoch/all_ai_models", meta["source_file"], header)
    snapshot = entry["sha256"]
    fields = sorted(header, key=str.lower)
    df = raw.withColumn("raw_payload", F.to_json(F.struct(*[F.col(sql_id(c)) for c in fields])))
    df = (df.withColumn("payload_hash", F.sha2("raw_payload", 256))
            .withColumn("snapshot_id", F.lit(snapshot))
            .withColumn("record_hash", F.sha2(F.concat_ws("|", F.lit(snapshot), F.col("raw_payload")), 256)))
    df = batch_columns(df.dropDuplicates(["record_hash"]), meta)
    df = df.select("raw_payload", "payload_hash", "record_hash", "snapshot_id", "source_file", "batch_id",
                   "collection_timestamp", "load_timestamp")
    step.rows_inserted, step.rows_updated = merge_insert_only(df, T_EPOCH)
    step.rows_skipped = step.rows_read - step.rows_inserted

### CELL 11 - The batch runner (fully parameter driven, one failure never stops the others)

In [ ]:
# CELL 11 - The batch runner (fully parameter driven, one failure never stops the others)

LOADERS = {"arena": load_arena, "cooperbench": load_cooperbench, "benchlm": load_benchlm, "epoch": load_epoch}


def find_manifests():
    folder = f"{SRC_DIR}/_manifests"
    found = []
    if not exists(folder):
        return found
    for f in dbutils.fs.ls(folder):
        if not (f.name.startswith("manifest__") and f.name.endswith(".json")):
            continue
        with open(local(f.path), encoding="utf-8") as handle:
            m = json.load(handle)
        m["_path"] = local(f.path)
        day = m["created_at"][:10]
        if P["load_type"] != "all" and m["load_type"] != P["load_type"]:
            continue
        if P["batch_id"] and m["batch_id"] != P["batch_id"]:
            continue
        bid = m["batch_id"]
        if FROM_B and bid[:len(FROM_B)] < FROM_B:
            continue
        if TO_B and bid[:len(TO_B)] > TO_B:
            continue
        if (START and day < START) or (END and day > END):
            continue
        found.append(m)
    return sorted(found, key=lambda m: (m["created_at"], m["batch_id"]))     # oldest first = arrival order


def volume_rule_ok(m):
    needed = MIN_FULL_BYTES if m["load_type"] == "full" else MIN_INCREMENTAL_BYTES
    return bool(m.get("gate_passed")) and int(m.get("landed_bytes", 0)) >= needed, needed


def already_loaded(rel, sha):
    q = spark.sql(f"SELECT 1 FROM {T_RUNS} WHERE layer = 'Raw-to-Bronze' AND status = 'SUCCESS' "
                  f"AND file_or_param = '{rel}' AND get_json_object(parameters_json, '$.sha256') = '{sha}' LIMIT 1")
    return q.count() > 0


def process_entry(m, e):
    """Returns LOADED | SKIPPED | QUARANTINED | FAILED | DONE"""
    rel = e["path"]
    src, processed, quarantined = f"{SRC_DIR}/{rel}", f"{ROOT}/processed/{rel}", f"{ROOT}/quarantine/{rel}"
    if not exists(src):
        if IS_LIVE and (exists(processed) or exists(quarantined)):
            return "DONE"
        log_error("Raw-to-Bronze", e["source"], rel, "File listed in the manifest was not found",
                  error_type="FileMissing", batch_id=m["batch_id"])
        return "FAILED"

    meta = {"batch_id": m["batch_id"], "load_type": m["load_type"], "created_at": m["created_at"],
            "dataset": e["dataset"], "source_file": rel}
    step = Step("Raw-to-Bronze", e["source"], e["dataset"], m["load_type"], rel, m["batch_id"],
                extra={"sha256": e["sha256"], "size_bytes": e["size_bytes"]})
    try:
        if os.path.getsize(local(src)) != e["size_bytes"] or sha256_file(src) != e["sha256"]:
            raise IOError("File size or SHA-256 does not match the manifest (damaged or incomplete upload)")
        step.bytes = e["size_bytes"]
        if not FORCE and already_loaded(rel, e["sha256"]):
            step.finish("SKIPPED", "Same file (same SHA-256) was already loaded successfully")
            if IS_LIVE:
                move(src, processed)
            return "SKIPPED"
        LOADERS[e["source"]](e, meta, src, step)
        step.finish("SUCCESS")
        if IS_LIVE:
            move(src, processed)
        return "LOADED"
    except QuarantineFile as q:
        step.rows_quarantined = max(step.rows_quarantined, 1)
        log_quarantine(m["batch_id"], e["source"], e["dataset"], rel, rel, "SCHEMA_CONTRACT", q, e["size_bytes"],
                       sample=q.sample)
        step.finish("QUARANTINED", q)
        if IS_LIVE:
            move(src, quarantined)
        return "QUARANTINED"
    except Exception as exc:
        log_error("Raw-to-Bronze", e["source"], rel, exc, error_type=type(exc).__name__,
                  batch_id=m["batch_id"], stack=traceback.format_exc())
        step.finish("FAILURE", exc)
        return "FAILED"


def run_pipeline():
    ingest_collector_logs()
    manifests = find_manifests()
    print("Batches to process:", len(manifests), "(oldest first)" + (" - REPLAY" if REPLAY else ""))
    for m in manifests:
        print("   ", m["created_at"], m["batch_id"], m["load_type"])
    totals = {"LOADED": 0, "SKIPPED": 0, "QUARANTINED": 0, "FAILED": 0, "DONE": 0, "REJECTED": 0}
    for m in manifests:
        batch_step = Step("Raw-to-Bronze (batch)", "all", m["batch_id"], m["load_type"], m["_path"], m["batch_id"])
        ok, needed = volume_rule_ok(m)
        if not ok:
            log_error("Raw-to-Bronze", "all", m["_path"],
                      f"Batch rejected: landed {m.get('landed_bytes')} bytes, rule needs {needed}",
                      error_type="VolumeRuleNotMet", batch_id=m["batch_id"])
            batch_step.finish("REJECTED", "Volume rule not met")
            totals["REJECTED"] += 1
            if IS_LIVE:                      # park it, so the same refusal is not logged on every run
                move(m["_path"], f"{ROOT}/quarantine/{os.path.basename(m['_path'])}")
            continue
        outcomes = []
        for e in m["files"]:
            if P["source"] != "all" and e["source"] != P["source"]:
                continue
            outcome = process_entry(m, e)
            outcomes.append(outcome)
            totals[outcome] += 1
        batch_step.bytes = int(m.get("landed_bytes", 0))
        batch_ok = "FAILED" not in outcomes
        batch_step.finish("SUCCESS" if batch_ok else "FAILURE", f"files: {len(outcomes)}")
        if IS_LIVE and batch_ok and P["source"] == "all":
            move(m["_path"], f"{ROOT}/processed/_manifests/{os.path.basename(m['_path'])}")
    RUN_TOTALS["outcomes"] = totals
    print("Finished:", totals)
    if totals["FAILED"]:
        raise RuntimeError(f"{totals['FAILED']} file(s) failed. See {T_ERRORS}. Re-run to retry only those.")

KEY_TABLES = {"arena": T_ARENA, "cooperbench": T_COOP, "benchlm": T_BLM, "epoch": T_EPOCH}


def check_primary_keys():
    """Data-quality check after loading: record_hash must be unique and never null in every Bronze table.
    One log row per table (layer 'Bronze data quality (primary key)'). Raises if a key problem exists."""
    problems = []
    for source, table in KEY_TABLES.items():
        step = Step("Bronze data quality (primary key)", source, table.split(".")[-1], "check",
                    "record_hash unique and not null", "")
        df = spark.table(table)
        total = df.count()
        duplicates = df.groupBy("record_hash").count().filter("count > 1").count()
        nulls = df.filter(F.col("record_hash").isNull()).count()
        step.rows_read = total
        ok = duplicates == 0 and nulls == 0
        step.finish("SUCCESS" if ok else "FAILURE", f"rows: {total}, duplicate keys: {duplicates}, null keys: {nulls}")
        if not ok:
            problems.append(f"{table}: {duplicates} duplicate and {nulls} null record_hash")
    if problems:
        raise RuntimeError("Primary key problem: " + "; ".join(problems))


def run_with_log():
    """Every run writes exactly one run-level row to ops.pipeline_execution_logs: success, failure, a bad
    parameter, or a run that found nothing to do. File and batch rows are written inside run_pipeline()."""
    run_step = Step("Raw-to-Bronze (run)", P["source"], "all", P["load_type"],
                    f"source_dir={SRC_DIR}" + (" | REPLAY" if REPLAY else ""),
                    P["batch_id"] or (f"{FROM_B or 'start'}..{TO_B or 'end'}" if (FROM_B or TO_B) else "all"))
    try:
        if PARAM_ERRORS:
            raise ValueError("Invalid parameter: " + "; ".join(PARAM_ERRORS))
        run_pipeline()
        if RUN_TOTALS["rows_inserted"] > 0 or REPLAY or FORCE:      # nothing new = nothing to re-check
            check_primary_keys()
        outcomes = RUN_TOTALS["outcomes"]
        note = "no batch matched the parameters, nothing to load" if not any(outcomes.values()) else f"files: {outcomes}"
        status = "SUCCESS"
    except Exception as exc:
        log_error("Raw-to-Bronze", P["source"], run_step.file_or_param, exc, error_type=type(exc).__name__,
                  batch_id=run_step.batch_id, stack=traceback.format_exc())
        status, note = "FAILURE", exc
        failure = exc
    else:
        failure = None
    run_step.rows_read, run_step.rows_inserted = RUN_TOTALS["rows_read"], RUN_TOTALS["rows_inserted"]
    run_step.rows_updated, run_step.rows_skipped = RUN_TOTALS["rows_updated"], RUN_TOTALS["rows_skipped"]
    run_step.rows_quarantined, run_step.bytes = RUN_TOTALS["rows_quarantined"], RUN_TOTALS["bytes"]
    run_step.finish(status, note)
    if failure is not None:
        raise failure


run_with_log()

### CELL 12 - Health check

In [ ]:
# CELL 12 - Health check

def _count(folder):
    return len(dbutils.fs.ls(folder)) if exists(folder) else 0


display(spark.createDataFrame(
    [("new (waiting to be loaded)", _count(f"{ROOT}/new/_manifests")),
     ("processed (already loaded, replayable)", _count(f"{ROOT}/processed/_manifests")),
     ("quarantine (set aside)", _count(f"{ROOT}/quarantine"))],
    "staging_area string, batches_or_items long"))
display(spark.sql(f"""
    SELECT 'arena_leaderboard_raw' AS table_name, COUNT(*) AS rows_loaded, MAX(load_timestamp) AS last_load FROM {T_ARENA}
    UNION ALL SELECT 'cooperbench_member_raw', COUNT(*), MAX(load_timestamp) FROM {T_COOP}
    UNION ALL SELECT 'benchlm_raw', COUNT(*), MAX(load_timestamp) FROM {T_BLM}
    UNION ALL SELECT 'epoch_models_raw', COUNT(*), MAX(load_timestamp) FROM {T_EPOCH}"""))
display(spark.sql(f"""SELECT layer, file_or_param, status, rows_read, rows_inserted, rows_updated,
    rows_skipped_duplicate, start_time, end_time FROM {T_RUNS} ORDER BY load_timestamp DESC LIMIT 20"""))
display(spark.sql(f"""SELECT quarantined_at, source, source_file, reason, detail, record_sample FROM {T_QUAR}
    ORDER BY quarantined_at DESC LIMIT 20"""))
display(spark.sql(f"""SELECT detected_at, dataset, column_name, event_type, expected_type, actual_type, action_taken
    FROM {T_DRIFT} ORDER BY detected_at DESC LIMIT 20"""))
display(spark.sql(f"""SELECT occurred_at, component, severity, source, file_or_param, error_type, error_message
    FROM {T_ERRORS} WHERE resolved = false ORDER BY occurred_at DESC LIMIT 20"""))

### CELL 13 (optional) - Re-run quarantined files after you fixed the cause

In [ ]:
# CELL 13 (optional) - Re-run quarantined files after you fixed the cause

RELEASE_FILES = []        # e.g. ["arena/text__full__20261010T031500Z.parquet"]  (empty = does nothing)

for rel in RELEASE_FILES:
    source_batch = None
    for f in dbutils.fs.ls(f"{ROOT}/processed/_manifests"):
        with open(local(f.path), encoding="utf-8") as handle:
            m = json.load(handle)
        if any(x["path"] == rel for x in m["files"]):
            source_batch = m
    if source_batch is None:
        print("No manifest found for", rel)
        continue
    entry = [x for x in source_batch["files"] if x["path"] == rel][0]
    move(f"{ROOT}/quarantine/{rel}", f"{ROOT}/new/{rel}")
    release = dict(source_batch, batch_id=source_batch["batch_id"] + "_release", files=[entry])
    with open(f"{ROOT}/new/_manifests/manifest__{release['batch_id']}.json", "w", encoding="utf-8") as handle:
        json.dump(release, handle, indent=2)
    print("Released:", rel, "- run the notebook again to load it.")